In [12]:

from google.colab import files

uploaded = files.upload()


KeyboardInterrupt: 

In [ ]:

import zipfile
import os

zip_name = "bioinformatics_splice_project.zip"
project_dir = "/content/research_project"

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(project_dir)

for root, dirs, filenames in os.walk(project_dir):
    for filename in filenames:
        print(os.path.join(root, filename))


In [ ]:

import os

project = "/content/research_project/bioinformatics_splice_project"

if os.path.isdir(project):
    print("Project folder found!")
    for root, dirs, files in os.walk(project):
        for name in files:
            print(os.path.join(root, name))
else:
    print("Project folder not found at the expected path.")
    print("Check the Files panel or Google Drive.")


In [ ]:

%cd /content/research_project

!find . -name requirements.txt


In [ ]:

!pip install -r /content/research_probioinformatics_splice_project/requirements.txt


In [ ]:

!python /content/research_project/bioinformatics_splice_project/run_analysis.py


In [ ]:

import os

for root, dirs, files_list in os.walk("/content/research_project"):
    for name in files_list:
        if name.endswith((".csv", ".png", ".pdf", ".md")):
            print(os.path.join(root, name))


In [ ]:

import os

print("Current folder:", os.getcwd())
print("\nFiles and folders:")

for item in os.listdir("/content"):
    print(item)


In [ ]:

import os

for root, dirs, files in os.walk("/content/research_project"):
    # Hidden/configuration folders ko skip karein
    dirs[:] = [d for d in dirs if d not in [".config", "__pycache__"]]

    level = root.replace("/content/research_project", "").count(os.sep)
    if level > 3:
        dirs[:] = []
        continue

    print("\nFolder:", root)

    for file in files:
        print("  -", file)


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/research_project/bioinformatics_splice_project")
results = base / "results"

for filename in [
    "test_metrics.csv",
    "cross_validation_metrics.csv",
    "class_distribution.csv"
]:
    path = results / filename
    print("\n" + "=" * 60)
    print(filename)

    if path.exists():
        print(pd.read_csv(path).to_string(index=False))
    else:
        print("File not found")

print("\n" + "=" * 60)
print("RESULTS SUMMARY")

summary = results / "results_summary.md"
if summary.exists():
    print(summary.read_text(encoding="utf-8"))

print("\n" + "=" * 60)
print("MANUSCRIPT DRAFT")

draft = base / "manuscript_draft.md"
if draft.exists():
    print(draft.read_text(encoding="utf-8")[:10000])


In [ ]:

from pathlib import Path

base = Path("/content/research_project/bioinformatics_splice_project")
results = base / "results"

for name in ["classification_reports.txt", "test_predictions.csv"]:
    path = results / name
    print("\n" + "=" * 55)
    print(name)

    if path.exists() and name.endswith(".txt"):
        print(path.read_text(encoding="utf-8")[:12000])
    elif path.exists():
        import pandas as pd
        df = pd.read_csv(path)
        print("Columns:", df.columns.tolist())
        print(df.head(10).to_string(index=False))


In [ ]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

base = Path("/content/research_project/bioinformatics_splice_project")
df = pd.read_csv(base / "results" / "test_predictions.csv")

labels = ["EI", "IE", "N"]

models = [
    "Nucleotide composition + logistic regression",
    "Position-specific one-hot + logistic regression",
    "Character k-mers (2–6) + linear SVM",
]

for model in models:
    print("\n" + "=" * 65)
    print(model)

    print("\nConfusion matrix (rows = true labels, columns = predicted labels):")
    print(pd.DataFrame(
        confusion_matrix(df["true_label"], df[model], labels=labels),
        index=labels,
        columns=labels
    ))

# Check for exact duplicate sequences across the full dataset.
# This is a screening check, not proof of train-test leakage.
data = pd.read_csv(base / "data" / "uci_splice_sequences.csv")

print("\n" + "=" * 65)
print("Dataset columns:", data.columns.tolist())
print("Dataset rows:", len(data))
print("Exact duplicate rows:", data.duplicated().sum())


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

base = Path("/content/research_project/bioinformatics_splice_project")
data = pd.read_csv(base / "data" / "uci_splice_sequences.csv")

data["sequence"] = data["sequence"].astype(str).str.strip().str.upper()
data["label"] = data["label"].astype(str).str.strip()

# Reconstruct the original stratified split
train_idx, test_idx = train_test_split(
    np.arange(len(data)),
    test_size=0.25,
    random_state=42,
    stratify=data["label"]
)

train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

train_sequences = set(train["sequence"])
test_sequences = set(test["sequence"])
overlap = train_sequences & test_sequences

# Exact duplicate rows
print("Total rows:", len(data))
print("Exact duplicate rows:", int(data.duplicated().sum()))
print("Train rows:", len(train))
print("Test rows:", len(test))

print("\nUnique sequences in both partitions:")
print("Train-test sequence overlap:", len(overlap))

print("\nTest rows whose sequence also appears in training:")
print(int(test["sequence"].isin(train_sequences).sum()))

# Check identical sequences carrying conflicting labels
labels_per_sequence = data.groupby("sequence")["label"].nunique()
conflicting_sequences = int((labels_per_sequence > 1).sum())

print("\nUnique sequences with conflicting labels:",
      conflicting_sequences)

# Show a few examples of overlap
print("\nExample overlapping sequences:")
print(list(overlap)[:5])


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/research_project/bioinformatics_splice_project")
df = pd.read_csv(base / "data" / "uci_splice_sequences.csv")

df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
df["label"] = df["label"].astype(str).str.strip()

# Find sequences associated with more than one label
label_counts = df.groupby("sequence")["label"].nunique()
conflicts = label_counts[label_counts > 1].index

print("Conflicting sequences:", len(conflicts))

for seq in conflicts:
    print("\nSequence:", seq)
    print(df[df["sequence"] == seq]["label"].value_counts().to_string())
    print("\nAll matching rows:")
    print(df[df["sequence"] == seq].to_string(index=False))

# Count sequence groups containing repeated rows
group_sizes = df.groupby("sequence").size()

print("\nUnique sequences:", df["sequence"].nunique())
print("Sequences occurring more than once:", int((group_sizes > 1).sum()))
print("Rows belonging to repeated-sequence groups:",
      int(group_sizes[group_sizes > 1].sum()))


In [ ]:

from pathlib import Path

script = Path(
    "/content/research_project/bioinformatics_splice_project/run_analysis.py"
)

print(script.read_text())


In [ ]:

from pathlib import Path

base = Path("/content/research_project/bioinformatics_splice_project")
source = base / "run_analysis.py"
target = base / "run_grouped_analysis.py"

code = source.read_text()

# Add StratifiedGroupKFold to the model-selection imports.
code = code.replace(
    "from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate",
    "from sklearn.model_selection import StratifiedGroupKFold, cross_validate"
)

# Replace main() with a duplicate-aware evaluation procedure.
start = code.index("def main():")
end = code.index('\nif __name__ == "__main__":', start)

new_main = r'''
def main():
    RESULTS.mkdir(parents=True, exist_ok=True)
    FIGURES.mkdir(parents=True, exist_ok=True)

    X, y = load_uci_dataset()
    X = X.astype(str).str.strip().str.upper()
    y = y.astype(str).str.strip().str.upper()
    groups = X.copy()  # Identical sequences must stay in the same fold.

    data = pd.DataFrame({"sequence": X, "label": y})
    conflict_counts = data.groupby("sequence")["label"].nunique()
    conflicting = conflict_counts[conflict_counts > 1].index
    pd.DataFrame({
        "sequence": list(conflicting),
        "labels": [
            "|".join(sorted(data.loc[data.sequence == seq, "label"].unique()))
            for seq in conflicting
        ],
        "rows": [
            int((data.sequence == seq).sum()) for seq in conflicting
        ],
    }).to_csv(RESULTS / "conflicting_sequence_audit.csv", index=False)

    print("Rows:", len(X))
    print("Unique sequences:", X.nunique())
    print("Conflicting sequence groups:", len(conflicting))
    print("Class counts:\n", y.value_counts().reindex(LABEL_ORDER, fill_value=0))

    # Four grouped folds: one fold is held out as test, the other three
    # form the development/training set. No sequence group crosses the split.
    outer_cv = StratifiedGroupKFold(
        n_splits=4, shuffle=True, random_state=RANDOM_STATE
    )
    train_idx, test_idx = next(outer_cv.split(X, y, groups=groups))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    train_groups = groups.iloc[train_idx]
    test_groups = groups.iloc[test_idx]

    overlap = set(train_groups).intersection(set(test_groups))
    if overlap:
        raise RuntimeError(f"Exact sequence leakage remains: {len(overlap)} sequences")

    print("Training rows:", len(X_train), "Test rows:", len(X_test))
    print("Train unique sequences:", X_train.nunique())
    print("Test unique sequences:", X_test.nunique())
    print("Train/test sequence overlap:", len(overlap))
    print("Test class counts:\n", y_test.value_counts().reindex(LABEL_ORDER, fill_value=0))

    counts = y.value_counts().reindex(LABEL_ORDER, fill_value=0)
    counts.rename_axis("class").reset_index(name="n").to_csv(
        RESULTS / "grouped_class_distribution.csv", index=False
    )

    models = make_models()
    # Group-aware CV only within the training/development data.
    cv = StratifiedGroupKFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE + 1
    )
    scoring = {
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "balanced_accuracy": "balanced_accuracy",
    }
    rows, cv_rows = [], []
    predictions = pd.DataFrame({
        "sequence": X_test.reset_index(drop=True),
        "true_label": y_test.reset_index(drop=True),
    })
    fitted = {}
    classes = LABEL_ORDER

    for name, model in models.items():
        print(f"\nTraining: {name}")
        cv_result = cross_validate(
            clone(model), X_train, y_train,
            cv=cv, groups=train_groups,
            scoring=scoring, n_jobs=-1, return_train_score=False
        )
        cv_row = {"model": name}
        for metric in scoring:
            values = cv_result[f"test_{metric}"]
            cv_row[f"cv_{metric}_mean"] = float(np.mean(values))
            cv_row[f"cv_{metric}_sd"] = float(np.std(values, ddof=1))
        cv_rows.append(cv_row)

        model.fit(X_train, y_train)
        fitted[name] = model
        pred, metrics = get_scores(model, X_test, y_test, classes)
        rows.append({"model": name, **metrics})
        predictions[name] = pred
        print(json.dumps({"model": name, **metrics}, indent=2))

    metrics_df = pd.DataFrame(rows)
    cv_df = pd.DataFrame(cv_rows)
    metrics_df.to_csv(RESULTS / "grouped_test_metrics.csv", index=False)
    cv_df.to_csv(RESULTS / "grouped_cross_validation_metrics.csv", index=False)
    predictions.to_csv(RESULTS / "grouped_test_predictions.csv", index=False)

    (RESULTS / "grouped_classification_reports.txt").write_text(
        "\n\n".join(
            f"{name}\n" + classification_report(
                y_test, fitted[name].predict(X_test),
                labels=classes, zero_division=0
            )
            for name in models
        ),
        encoding="utf-8"
    )

    # Confusion matrix for the k-mer model.
    km_name = "Character k-mers (2–6) + linear SVM"
    cm = confusion_matrix(
        y_test, fitted[km_name].predict(X_test), labels=LABEL_ORDER
    )
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=LABEL_ORDER
    ).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title("K-mer model: sequence-grouped test confusion matrix")
    fig.tight_layout()
    fig.savefig(FIGURES / "figure3_grouped_kmer_confusion_matrix.png", dpi=300)
    plt.close(fig)

    # Grouped held-out metrics comparison.
    plot_df = metrics_df.set_index("model")[
        ["accuracy", "balanced_accuracy", "macro_f1"]
    ]
    ax = plot_df.plot(kind="bar", figsize=(10.5, 5.2), ylim=(0, 1))
    ax.set(title="Sequence-grouped held-out performance",
           xlabel="Model", ylabel="Score")
    ax.tick_params(axis="x", rotation=15)
    ax.legend(loc="lower right")
    ax.figure.tight_layout()
    ax.figure.savefig(FIGURES / "figure2_grouped_test_metrics.png", dpi=300)
    plt.close(ax.figure)

    summary = [
        "# Sequence-grouped evaluation summary", "",
        "Dataset: UCI Molecular Biology (Splice-junction Gene Sequences), ID 69.",
        f"Rows: {len(X)}; unique sequences: {X.nunique()}; sequence length: 60.",
        f"Train rows: {len(X_train)}; test rows: {len(X_test)}.",
        f"Train unique sequences: {X_train.nunique()}; test unique sequences: {X_test.nunique()}.",
        f"Exact sequence overlap across train/test: {len(overlap)}.",
        f"Conflicting-label sequence groups retained and audited: {len(conflicting)}.",
        "Outer split: first fold of 4-fold StratifiedGroupKFold; random seed 42.",
        "Cross-validation: 5-fold StratifiedGroupKFold within training data only.", "",
        "## Grouped held-out test results", "",
        metrics_df.round(4).to_markdown(index=False), "",
        "## Grouped cross-validation results", "",
        cv_df.round(4).to_markdown(index=False), "",
        "## Interpretation and limitations", "",
        "Identical sequence strings are kept within a single split. The conflicting-label sequence is retained and documented rather than silently removed. Grouped stratification may not yield perfectly balanced classes. Exact-sequence grouping does not prevent leakage from near-identical or homologous sequences. These results do not establish generalization to independent genomes or clinical samples."
    ]
    (RESULTS / "grouped_results_summary.md").write_text(
        "\n".join(summary), encoding="utf-8"
    )

    print("\nGrouped evaluation complete.")
    print("Summary:", RESULTS / "grouped_results_summary.md")
    print("Metrics:", RESULTS / "grouped_test_metrics.csv")
'''

code = code[:start] + new_main + code[end:]
target.write_text(code, encoding="utf-8")
print("Created:", target)
print("Original run_analysis.py remains unchanged.")


In [ ]:

%cd /content/research_project/bioinformatics_splice_project
!python run_grouped_analysis.py


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/research_project/bioinformatics_splice_project")
results = base / "results"

print("=== GROUPED CROSS-VALIDATION ===")
cv = pd.read_csv(results / "grouped_cross_validation_metrics.csv")
print(cv.round(4).to_string(index=False))

print("\n=== GROUPED TEST METRICS ===")
test = pd.read_csv(results / "grouped_test_metrics.csv")
print(test.round(4).to_string(index=False))

print("\n=== CONFLICTING SEQUENCE AUDIT ===")
audit = pd.read_csv(results / "conflicting_sequence_audit.csv")
print(audit.to_string(index=False))

print("\n=== UNIQUE-SEQUENCE TEST AUDIT ===")
pred = pd.read_csv(results / "grouped_test_predictions.csv")
print("Test rows:", len(pred))
print("Unique test sequences:", pred["sequence"].nunique())
print("Repeated test rows:", len(pred) - pred["sequence"].nunique())
print("Test sequences with conflicting labels in full dataset:",
      pred["sequence"].isin(audit["sequence"]).sum())

print("\n=== GENERATED SUMMARY ===")
print((results / "grouped_results_summary.md").read_text())


In [ ]:

from pathlib import Path
import pandas as pd
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef, classification_report
)

base = Path("/content/research_project/bioinformatics_splice_project")
results = base / "results"

pred = pd.read_csv(results / "grouped_test_predictions.csv")

# Verify that each test sequence has a single true label.
label_counts = pred.groupby("sequence")["true_label"].nunique()
conflicting_test = label_counts[label_counts > 1]

print("Unique test sequences:", pred["sequence"].nunique())
print("Sequences with conflicting test labels:", len(conflicting_test))

if len(conflicting_test):
    raise ValueError(
        "Conflicting test labels found; resolve this before unique-sequence scoring."
    )

# Retain one row per unique sequence.
unique_pred = pred.drop_duplicates(subset="sequence", keep="first").copy()

model_columns = [
    "Nucleotide composition + logistic regression",
    "Position-specific one-hot + logistic regression",
    "Character k-mers (2–6) + linear SVM",
]

rows = []
for model in model_columns:
    y_true = unique_pred["true_label"]
    y_hat = unique_pred[model]

    rows.append({
        "model": model,
        "unique_test_sequences": len(unique_pred),
        "accuracy": accuracy_score(y_true, y_hat),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_hat),
        "macro_precision": precision_score(
            y_true, y_hat, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_hat, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_hat, average="macro", zero_division=0
        ),
        "mcc": matthews_corrcoef(y_true, y_hat),
    })

unique_metrics = pd.DataFrame(rows)
unique_metrics.to_csv(
    results / "unique_sequence_test_metrics.csv", index=False
)

print("\n=== UNIQUE-SEQUENCE TEST METRICS ===")
print(unique_metrics.round(4).to_string(index=False))

print("\n=== BEST MODEL REPORT ===")
best_model = "Position-specific one-hot + logistic regression"
print(classification_report(
    unique_pred["true_label"],
    unique_pred[best_model],
    labels=["EI", "IE", "N"],
    zero_division=0
))
print("\nSaved:", results / "unique_sequence_test_metrics.csv")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/research_project/bioinformatics_splice_project")
r = base / "results"

# Combine the already-computed results into one comparison table.
random = pd.read_csv(r / "test_metrics.csv")
grouped = pd.read_csv(r / "grouped_test_metrics.csv")
unique = pd.read_csv(r / "unique_sequence_test_metrics.csv")

comparison = (
    random[["model", "accuracy", "macro_f1"]]
    .rename(columns={"accuracy": "random_accuracy",
                     "macro_f1": "random_macro_f1"})
    .merge(
        grouped[["model", "accuracy", "macro_f1"]].rename(
            columns={"accuracy": "grouped_accuracy",
                     "macro_f1": "grouped_macro_f1"}
        ), on="model"
    )
    .merge(
        unique[["model", "accuracy", "macro_f1"]].rename(
            columns={"accuracy": "unique_accuracy",
                     "macro_f1": "unique_macro_f1"}
        ), on="model"
    )
)

comparison.to_csv(r / "final_model_comparison.csv", index=False)
print(comparison.round(4).to_string(index=False))
print("\nSaved:", r / "final_model_comparison.csv")
print("\nBest model: Position-specific one-hot + logistic regression")
print("Unique-sequence test accuracy: 94.27%")
print("Unique-sequence test macro-F1: 93.28%")


In [ ]:

from pathlib import Path
import shutil

base = Path("/content/research_project/bioinformatics_splice_project")
backup = Path("/content/bioinformatics_splice_project_backup")

if not base.exists():
    raise FileNotFoundError(f"Project folder nahi mila: {base}")

if backup.exists():
    shutil.rmtree(backup)

shutil.copytree(base, backup)

print("Backup successfully created!")
print("Backup location:", backup)
print("Files backed up:", sum(1 for p in backup.rglob("*") if p.is_file()))


In [ ]:

import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/bioinformatics_splice_project_backup",
    "zip",
    "/content/bioinformatics_splice_project_backup"
)

files.download(zip_path)


In [ ]:

from pathlib import Path

base = Path("/content/research_project/bioinformatics_splice_project")

for name in [
    "manuscript_draft.md",
    "manuscript_draft.docx",
    "README.md",
    "requirements.txt",
    "results/final_model_comparison.csv",
    "results/grouped_test_metrics.csv",
    "results/grouped_cross_validation_metrics.csv",
]:
    p = base / name
    print(("FOUND  " if p.exists() else "MISSING"), name)


In [ ]:

from pathlib import Path

base = Path("/content/research_project/bioinformatics_splice_project")

print("MANUSCRIPT FILES")
for p in base.glob("manuscript_draft.*"):
    print(p.name, "|", round(p.stat().st_size / 1024, 2), "KB")

print("\nRESULTS AND FIGURES")
for folder in ["results", "figures"]:
    path = base / folder
    if path.exists():
        for p in sorted(path.iterdir()):
            print(folder + "/" + p.name)


In [ ]:

from pathlib import Path
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

base = Path("/content/research_project/bioinformatics_splice_project")
results = base / "results"
figures = base / "figures"
output = base / "manuscript_final_results.docx"

# Read existing manuscript draft
md_path = base / "manuscript_draft.md"
if not md_path.exists():
    raise FileNotFoundError("manuscript_draft.md nahi mili.")

doc = Document()
section = doc.sections[0]
section.top_margin = Inches(0.8)
section.bottom_margin = Inches(0.8)
section.left_margin = Inches(1)
section.right_margin = Inches(1)

styles = doc.styles
styles["Normal"].font.name = "Times New Roman"
styles["Normal"].font.size = Pt(11)

# Convert existing Markdown text into Word paragraphs/headings
text = md_path.read_text(encoding="utf-8")
for line in text.splitlines():
    line = line.strip()
    if not line:
        continue
    if line.startswith("### "):
        doc.add_heading(line[4:], level=3)
    elif line.startswith("## "):
        doc.add_heading(line[3:], level=2)
    elif line.startswith("# "):
        doc.add_heading(line[2:], level=1)
    elif line.startswith("- ") or line.startswith("* "):
        doc.add_paragraph(line[2:], style="List Bullet")
    elif line.startswith("|"):
        # Markdown tables are added separately from the result CSVs below.
        continue
    else:
        doc.add_paragraph(line)

doc.add_page_break()
doc.add_heading("Experimental Results: Model Comparison", level=1)

doc.add_paragraph(
    "The models were evaluated using a grouped train-test split in which "
    "identical sequence strings were kept within the same partition. "
    "This reduces exact-sequence leakage between training and test sets. "
    "It does not exclude possible similarity between related sequences."
)

# Add the actual model comparison table
comparison_path = results / "final_model_comparison.csv"
if comparison_path.exists():
    df = pd.read_csv(comparison_path).round(4)
    doc.add_heading("Comparison across evaluation strategies", level=2)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Table Grid"

    for i, col in enumerate(df.columns):
        table.rows[0].cells[i].text = str(col.replace("_", " ").title())

    for _, row in df.iterrows():
        cells = table.add_row().cells
        for i, value in enumerate(row):
            cells[i].text = str(value)

# Add grouped test metrics from actual CSV
metrics_path = results / "grouped_test_metrics.csv"
if metrics_path.exists():
    df = pd.read_csv(metrics_path).round(4)
    doc.add_heading("Grouped test-set performance", level=2)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Table Grid"
    for i, col in enumerate(df.columns):
        table.rows[0].cells[i].text = str(col.replace("_", " ").title())
    for _, row in df.iterrows():
        cells = table.add_row().cells
        for i, value in enumerate(row):
            cells[i].text = str(value)

# Add grouped cross-validation results
cv_path = results / "grouped_cross_validation_metrics.csv"
if cv_path.exists():
    df = pd.read_csv(cv_path).round(4)
    doc.add_heading("Grouped cross-validation results", level=2)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Table Grid"
    for i, col in enumerate(df.columns):
        table.rows[0].cells[i].text = str(col.replace("_", " ").title())
    for _, row in df.iterrows():
        cells = table.add_row().cells
        for i, value in enumerate(row):
            cells[i].text = str(value)

# Insert available figures with captions
figure_list = [
    ("figure1_class_distribution.png",
     "Figure 1. Distribution of splice-junction sequence classes."),
    ("figure2_grouped_test_metrics.png",
     "Figure 2. Performance of the evaluated models on the grouped test set."),
    ("figure3_grouped_kmer_confusion_matrix.png",
     "Figure 3. Confusion matrix for the character k-mer model.")
]

doc.add_page_break()
doc.add_heading("Figures", level=1)

for filename, caption in figure_list:
    path = figures / filename
    if path.exists():
        doc.add_picture(str(path), width=Inches(5.8))
        p = doc.add_paragraph(caption)
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Document limitations and reproducibility note
doc.add_heading("Interpretation and Limitations", level=1)
doc.add_paragraph(
    "Position-specific one-hot encoding with logistic regression achieved "
    "the strongest performance among the evaluated representations in the "
    "reported grouped evaluation. This result indicates that nucleotide "
    "identity at individual sequence positions is informative for this dataset; "
    "it does not establish a biological mechanism or prove generalization to "
    "independent genomic datasets. The dataset contains repeated sequences and "
    "one sequence with conflicting labels, which was documented in the audit "
    "file. Grouped evaluation prevents identical sequence strings from crossing "
    "the train-test boundary, but near-identical or homologous sequences may "
    "still occur across partitions."
)

doc.save(output)
print("Word manuscript created:", output)
print("Size (KB):", round(output.stat().st_size / 1024, 1))


In [ ]:

%pip install python-docx


In [ ]:
from docx import Document

In [ ]:

from google.colab import files
files.download(
    "/content/research_project/bioinformatics_splice_project/manuscript_final_results.docx"
)


In [ ]:

from docx import Document
from pathlib import Path

path = Path(
    "/content/research_project/bioinformatics_splice_project/"
    "manuscript_final_results.docx"
)

doc = Document(path)

for paragraph in doc.paragraphs:
    if paragraph.text.strip():
        print(paragraph.text)

print("\n--- TABLES ---")
for i, table in enumerate(doc.tables, start=1):
    print(f"\nTable {i}")
    for row in table.rows:
        print(" | ".join(cell.text for cell in row.cells))


In [ ]:

from pathlib import Path
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

base = Path("/content/research_project/bioinformatics_splice_project")
results = base / "results"
figures = base / "figures"
out = base / "manuscript_revised.docx"

doc = Document()
sec = doc.sections[0]
sec.top_margin = Inches(0.8)
sec.bottom_margin = Inches(0.8)
sec.left_margin = Inches(0.85)
sec.right_margin = Inches(0.85)

doc.styles["Normal"].font.name = "Times New Roman"
doc.styles["Normal"].font.size = Pt(10.5)

def paragraph(text):
    doc.add_paragraph(text)

def heading(text, level=1):
    doc.add_heading(text, level=level)

def add_csv_table(filename, title):
    path = results / filename
    if not path.exists():
        paragraph(f"[Table not added: {filename} not found]")
        return
    df = pd.read_csv(path).fillna("Not calculated")
    heading(title, 2)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Table Grid"
    for i, col in enumerate(df.columns):
        table.rows[0].cells[i].text = str(col).replace("_", " ").title()
    for _, row in df.iterrows():
        cells = table.add_row().cells
        for i, value in enumerate(row):
            if isinstance(value, float):
                cells[i].text = f"{value:.4f}"
            else:
                cells[i].text = str(value)

def add_figure(filename, caption):
    path = figures / filename
    if path.exists():
        doc.add_picture(str(path), width=Inches(5.8))
        p = doc.add_paragraph(caption)
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER

# Title page
p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run(
    "Comparing Nucleotide Composition, Position-Specific Encoding, "
    "and K-mer Features for Splice-Junction Classification"
)
r.bold = True
r.font.size = Pt(16)

p = doc.add_paragraph()
p.alignment = WD_ALIGN_PARAGRAPH.CENTER
p.add_run("\nRayyan Anwar\n").bold = True
p.add_run("Affiliation: To be completed by the author")

heading("Abstract")
paragraph(
    "Splice-junction classification is a benchmark task in computational "
    "biology because sequence patterns around exon–intron boundaries provide "
    "information relevant to RNA splicing. This study compares three "
    "conventional machine-learning pipelines on the UCI Molecular Biology "
    "Splice-junction Gene Sequences dataset: nucleotide composition with "
    "logistic regression, position-specific one-hot encoding with logistic "
    "regression, and character TF–IDF k-mers with a linear support vector "
    "machine. The dataset contains 3,190 fixed-length sequence records with "
    "three labels: EI, IE, and N. To reduce exact-duplicate leakage, the "
    "primary evaluation groups identical sequence strings into the same "
    "partition. Position-specific encoding achieved a grouped test accuracy "
    "of 0.9427 and macro-F1 of 0.9353. Its five-fold grouped cross-validation "
    "accuracy was 0.9321 ± 0.0093, and macro-F1 was 0.9266 ± 0.0082. "
    "The composition baseline and k-mer model achieved lower scores under "
    "the same grouped evaluation. These findings are specific to this "
    "historical benchmark and do not establish generalization to independent "
    "genomes or clinical applications. Near-homologous sequence leakage and "
    "the dataset's documented conflicting-label record remain limitations."
)
paragraph(
    "Keywords: bioinformatics; splice junction; DNA sequence classification; "
    "k-mer; one-hot encoding; logistic regression; support vector machine."
)

heading("1. Introduction")
paragraph(
    "Pre-mRNA splicing removes introns and joins exons to produce mature RNA. "
    "Sequence patterns surrounding exon–intron boundaries can help distinguish "
    "splice-junction classes from other sequence windows. Computational "
    "classification of such windows provides a useful setting for comparing "
    "representations of nucleotide sequences."
)
paragraph(
    "Global nucleotide composition summarizes how frequently each base occurs "
    "but discards positional information. Position-specific encoding retains "
    "the nucleotide identity at each sequence coordinate. Character k-mer "
    "features represent short subsequences and can capture local patterns. "
    "These representations provide different assumptions about which aspects "
    "of a sequence are informative."
)
paragraph(
    "This study compares these three feature representations on the UCI "
    "Molecular Biology splice-junction dataset. The objective is a transparent "
    "baseline comparison rather than a claim of a new biological mechanism "
    "or state-of-the-art genomic prediction."
)

heading("2. Research question")
paragraph(
    "How do nucleotide-composition counts, position-specific nucleotide "
    "identity, and character k-mers differ in three-class splice-junction "
    "classification when evaluated with identical-sequence groups kept "
    "within the same data partition?"
)

heading("3. Materials and methods")
heading("3.1 Dataset", 2)
paragraph(
    "The dataset was retrieved using the UCI Machine Learning Repository "
    "interface (dataset ID 69). It contains 3,190 records of 60 nucleotide "
    "positions labelled EI, IE, or N. The analysis retains the sequence "
    "symbols as supplied after standard whitespace and letter-case "
    "normalization. Class counts and duplicate-sequence information are "
    "reported in the generated analysis outputs."
)
heading("3.2 Data partitioning", 2)
paragraph(
    "The primary evaluation uses StratifiedGroupKFold with four folds and "
    "random seed 42. Identical sequence strings are used as groups so that "
    "an exact sequence cannot occur in both the training and held-out test "
    "partitions. The first fold is used as the test set. Five-fold "
    "StratifiedGroupKFold cross-validation is performed within the training "
    "partition. A separate random stratified split was also evaluated for "
    "comparison. Because these strategies can produce different partitions "
    "and class distributions, score differences between them should not be "
    "interpreted as a causal estimate of leakage."
)
heading("3.3 Models", 2)
paragraph(
    "The composition baseline uses nucleotide-count features and "
    "class-balanced logistic regression. The position-specific model "
    "one-hot encodes nucleotide identity at each position and applies "
    "logistic regression. The k-mer model uses character TF–IDF features "
    "with n-gram lengths from two through six and a linear support vector "
    "machine. Vectorization is fitted within each model pipeline to prevent "
    "the test partition from determining the feature vocabulary."
)
heading("3.4 Evaluation metrics", 2)
paragraph(
    "Models are evaluated using accuracy, balanced accuracy, macro-precision, "
    "macro-recall, macro-F1, and Matthews correlation coefficient (MCC). "
    "Grouped cross-validation results are summarized using the mean and "
    "standard deviation across folds. ROC-AUC is not reported in the tables "
    "because the current saved metrics contain missing values for this "
    "measure; it should only be added after a valid score-based calculation."
)
heading("3.5 Reproducibility and data audit", 2)
paragraph(
    "The workflow writes metrics, predictions, classification reports, "
    "figures, and a conflicting-sequence audit to disk. The dataset contains "
    "3,190 records and 3,005 unique sequence strings. The audit identifies "
    "one sequence associated with conflicting labels. This record was "
    "retained and documented rather than silently removed. Exact-sequence "
    "grouping does not rule out leakage between highly similar or homologous "
    "sequences."
)

heading("4. Results")
paragraph(
    "The position-specific one-hot logistic regression model achieved the "
    "highest performance among the three evaluated models in the grouped "
    "test set. Its accuracy was 0.9427, balanced accuracy was 0.9445, "
    "macro-F1 was 0.9353, and MCC was 0.9066. The grouped five-fold "
    "cross-validation accuracy was 0.9321 ± 0.0093, with macro-F1 of "
    "0.9266 ± 0.0082. The k-mer model achieved grouped test accuracy of "
    "0.7248 and macro-F1 of 0.7031. The composition baseline achieved "
    "grouped test accuracy of 0.4919 and macro-F1 of 0.4896."
)
paragraph(
    "The position-specific model also achieved accuracy of 0.9427 and "
    "macro-F1 of 0.9328 when evaluated on the unique sequences in the "
    "grouped test set. This provides a duplicate-reduced sensitivity check, "
    "but it is not an independent external validation."
)

add_csv_table("final_model_comparison.csv",
              "Table 1. Performance across evaluation strategies")
add_csv_table("grouped_test_metrics.csv",
              "Table 2. Grouped held-out test performance")
add_csv_table("grouped_cross_validation_metrics.csv",
              "Table 3. Grouped cross-validation performance")
add_csv_table("unique_sequence_test_metrics.csv",
              "Table 4. Unique-sequence test sensitivity analysis")

heading("5. Discussion")
paragraph(
    "The position-specific representation outperformed both the composition "
    "baseline and the character k-mer representation in the reported "
    "evaluations. This pattern is consistent with nucleotide identity at "
    "individual sequence positions being informative for this benchmark. "
    "The results do not establish a biological mechanism, and the comparison "
    "does not isolate the causal contribution of any individual nucleotide "
    "position."
)
paragraph(
    "Grouped splitting reduces the risk that exact duplicate sequences "
    "inflate held-out performance. However, related sequences may still "
    "occur across partitions, and the historical dataset may not represent "
    "current genomic diversity or annotation practice. The scores should "
    "therefore be interpreted as benchmark-specific."
)

heading("6. Limitations")
paragraph(
    "Limitations include the age and size of the dataset, fixed-length "
    "sequence windows, a documented conflicting-label sequence, potential "
    "similarity leakage between homologous sequences, and the absence of "
    "external validation. The study compares three conventional model "
    "families and does not exhaust the range of modern sequence-learning "
    "methods. Independent validation and similarity-aware partitioning "
    "would strengthen future work."
)

heading("7. Conclusion")
paragraph(
    "In this UCI splice-junction benchmark, position-specific one-hot "
    "encoding with logistic regression achieved the strongest results among "
    "the evaluated representations. The grouped evaluation reduces exact "
    "duplicate leakage, but further validation using similarity-aware splits "
    "and an independent dataset is required before broader claims about "
    "genomic prediction can be made."
)

heading("Data and code availability")
paragraph(
    "The dataset is available from the UCI Machine Learning Repository, "
    "dataset ID 69, DOI: https://doi.org/10.24432/C5M888. The authors should "
    "deposit the analysis code, environment specification, data-processing "
    "details, and generated results in a public repository before submission."
)

heading("References")
paragraph(
    "1. UCI Machine Learning Repository. Molecular Biology (Splice-junction "
    "Gene Sequences). Dataset 69. https://doi.org/10.24432/C5M888"
)
paragraph(
    "2. Pedregosa F, Varoquaux G, Gramfort A, et al. Scikit-learn: Machine "
    "Learning in Python. Journal of Machine Learning Research. 2011;12:2825–2830. "
    "https://www.jmlr.org/papers/v12/pedregosa11a.html"
)
paragraph(
    "3. scikit-learn developers. Cross-validation: evaluating estimator "
    "performance. https://scikit-learn.org/stable/modules/cross_validation.html"
)
paragraph(
    "4. scikit-learn developers. Classification metrics. "
    "https://scikit-learn.org/stable/modules/model_evaluation.html"
)

heading("Figures")
add_figure("figure1_class_distribution.png",
           "Figure 1. Distribution of splice-junction sequence classes.")
add_figure("figure2_grouped_test_metrics.png",
           "Figure 2. Model performance on the grouped held-out test set.")
add_figure("figure3_grouped_kmer_confusion_matrix.png",
           "Figure 3. Confusion matrix for the character k-mer model.")

doc.save(out)
print("Created:", out)
print("File size (KB):", round(out.stat().st_size / 1024, 1))


In [ ]:

from google.colab import files
files.download(
    "/content/research_project/bioinformatics_splice_project/manuscript_revised.docx"
)


In [ ]:

import os

paths_to_check = [
    "/content/research_project/bioinformatics_splice_project",
    "/content/drive/MyDrive/research_project/bioinformatics_splice_project",
    "/content/drive/MyDrive/bioinformatics_splice_project",
]

for path in paths_to_check:
    print(f"\nChecking: {path}")
    if os.path.isdir(path):
        print("FOUND! Files:")
        for name in os.listdir(path):
            print(" -", name)
    else:
        print("Not found at this location.")


In [ ]:

import os

for base in ["/content", "/content/drive/MyDrive"]:
    if os.path.exists(base):
        print(f"\nSearching under: {base}")
        for root, dirs, files in os.walk(base):
            dirs[:] = [
                d for d in dirs
                if d not in {".git", "__pycache__", "node_modules"}
            ]
            if "bioinformatics_splice_project" in dirs:
                print("FOUND:", os.path.join(
                    root, "bioinformatics_splice_project"
                ))
                break


In [ ]:

import os

search_roots = ["/content"]

keywords = [
    "bioinformatics_splice_project",
    "run_grouped_analysis.py",
    "final_model_comparison.csv",
    "manuscript_draft.docx",
    "grouped_test_metrics.csv"
]

for base in search_roots:
    print("\nSearching:", base)
    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "__pycache__", "node_modules"}
        ]

        for name in dirs + files:
            if any(k.lower() in name.lower() for k in keywords):
                print(os.path.join(root, name))


In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

import os

root = "/content/drive/MyDrive"

for current, dirs, files in os.walk(root):
    dirs[:] = [
        d for d in dirs
        if d not in {".git", "__pycache__", "node_modules"}
    ]

    for name in dirs + files:
        if any(term in name.lower() for term in [
            "splice",
            "bioinformatics",
            "final_model_comparison",
            "manuscript_draft",
            "run_grouped_analysis"
        ]):
            print(os.path.join(current, name))
